Bruker er kurert liste fra "To Sultne Piger" som er publisert gjennom google. Henter ut all relevant data og genererer en csv med informasjonen fra listen.

In [36]:
%pip install pandas
%pip install pydantic
%pip install openai
%pip install requests
%pip install bs4
%pip install selenium


[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: pip install --upgr

In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd

def parse_kml(kml_file):
    # Read KMS
    tree = ET.parse(kml_file)
    root = tree.getroot()

    # DEfine namespace    
    ns = {'kml': 'http://www.opengis.net/kml/2.2'}
    
    placemarks = []
    
    # Get all "Placemark" items in the file
    for placemark in root.findall('.//kml:Placemark', ns):
        name_elem = placemark.find('kml:name', ns)
        desc_elem = placemark.find('kml:description', ns)
        # TODO: Address does not work, need to use cordinates insted
        address_elem = placemark.find('kml:address', ns)
        
        name = name_elem.text.strip() if name_elem is not None and name_elem.text else ''
        description = desc_elem.text.strip() if desc_elem is not None and desc_elem.text else ''
        address = address_elem.text.strip() if address_elem is not None and address_elem.text else ''
        
        # Dictionary for PD Indexes
        placemark_data = {
            'name': name,
            'raw_description': description,
            'description': '',       # Empty field
            'rating': '',           # Empty field
            'traits': '',            # Empty field
            'price': '',             # Empty field
            'cuisine': '',           # Empty field            
            'type': '',             # Empty field
            'instagram': '',        # Empty field
            'website': '',           # Empty field
            'phone': '',             # Empty field
            'email': '',             # Empty field
            'booking platform': '', # Empty field
            'address': address
        }
        placemarks.append(placemark_data)
        
    return placemarks

# KML File path
kml_file_path = 'copenhagen_curated_list.kml'
data = parse_kml(kml_file_path)

# Create Pandas DataFrame
df = pd.DataFrame(data)

# Write DF to file
csv_file_path = 'copenhagen_curated_list.csv'
df.to_csv(csv_file_path, index=False)

print(f"CSV file created: {csv_file_path}")

CSV file created: copenhagen_curated_list.csv


Scan gjennom alle description feltene som har blitt hentet ut tidligere. Hent ut rating og plasser de i rating kolonnen.

In [ ]:
import re

def extract_ratings(df):
    """
    For each row in the DataFrame, search the 'description' field for all occurrences
    of ratings in the format 'number/10' (e.g., '8/10', '7.5/10') and update the 'rating'
    column with a comma-separated string of all found ratings.
    
    Parameters:
        df (pandas.DataFrame): DataFrame containing a 'description' field.
        
    Returns:
        pandas.DataFrame: The modified DataFrame with the 'rating' field updated.
    """
    # Loop through each row
    for index, row in df.iterrows():
        description = row.get('raw_description', '')
        # Use Regex to get all numbers from raw description (each post has multiple rankings)
        matches = re.findall(r'(\d+(?:\.\d+)?)\s*/\s*10', description)
        # Join all matches to a string OBS: ranking always x/10
        df.at[index, 'rating'] = ", ".join(matches) if matches else ""
    return df

def extract_instagram_links(df):
    """
    For each row in the DataFrame, search the 'raw_description' field for all occurrences
    of Instagram links starting with 'https://www.instagram.com' and update the 'instagram'
    column with a comma-separated string of all found links.
    
    Parameters:
        df (pandas.DataFrame): DataFrame containing a 'description' field.
        
    Returns:
        pandas.DataFrame: The modified DataFrame with the 'instagram' column added.
    """
    
    for index, row in df.iterrows():
        description = row.get("raw_description", "")
        # Regex pattern: match links that start with https://www.instagram.com
        matches = re.findall(r'(https?://www\.instagram\.com[^\s\'"]+)', description)
        # If matches found, join them into a comma-separated string
        df.at[index, "instagram"] = ", ".join(matches) if matches else ""
    return df


df = extract_ratings(df)
df = extract_instagram_links(df)
df.to_csv('updated_copenhagen_curated_list.csv', index=False)

Use LLM to extract data from the raw_description field, and populate other fields in the CSV. Use gpt model 4o-mini and structured outputs

In [ ]:
from openai import OpenAI
import os
import json
import time
from pydantic import BaseModel

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
)

class RestaurantInfo(BaseModel):
    type: str
    traits: str
    cuisine: str
    description: str
    price: str

def extract_structured_info(df, model="gpt-4o-mini", max_entries=5):
    """
    For the first `max_entries` rows of the DataFrame, extract structured information from the
    'raw_description' field using the OpenAI API. The API is expected to return a JSON in the format:
    
      {
          "type": "value",
          "traits": "tag1, tag2, tag3",
          "cuisine": "value"
      }
      
    The function updates/creates the following DataFrame columns: 'description', 'type', 'traits', and 'cuisine'.
    The 'description' field is populated with the original raw description.
    
    Parameters:
        df (pandas.DataFrame): Input DataFrame with a 'raw_description' column.
        model (str): The OpenAI model to use (default is "gpt-4o-mini").
        max_entries (int): Number of rows to process (default is 5).
        
    Returns:
        pandas.DataFrame: The updated DataFrame.
    """
    for idx in df.index[:max_entries]:
        raw_description = df.at[idx, 'raw_description']
        # Construct the prompt for the API call
        prompt = (
            "Analyser den følgende danske restaurantbeskrivelse og træk ud følgende information:\n"
            "- Description: beskrivelsen af restauranten\n"
            "- Type: fx lunch, drinks, dinner osv.\n"
            "- Price: prisleje (fx DKK)\n"
            "- Traits: nøgleord der beskriver oplevelsen (fx great vibes, upbeat, great food)\n"
            "- Cuisine: hvilken type mad der serveres\n\n"
            "Returnér svaret som et JSON-objekt med nøglerne 'type', 'traits' og 'cuisine'.\n\n"
            f"Beskrivelse:\n{raw_description}"
        )
        messages = [
            {"role": "system", "content": "Du er en hjælpsom assistent, der ekstraherer struktureret information fra danske restaurantbeskrivelser."},
            {"role": "user", "content": prompt}
        ]
        
        try:
            # Call the API using structured output with the Pydantic model
            completion = client.beta.chat.completions.parse(
                model=model,
                messages=messages,
                temperature=0,  # Use deterministic output
                response_format=RestaurantInfo,
            )
            # The parsed response is already a RestaurantInfo object
            info = completion.choices[0].message.parsed
            print(f"Response for row {idx}:\n{info}")
        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            # In case of error, assign empty strings
            info = RestaurantInfo(type="", traits="", cuisine="", description="", price="")
        
        # Update the DataFrame:
        # - 'description' field gets the original raw description
        # - 'type', 'traits', and 'cuisine' come from the API response
        df.at[idx, 'description'] = info.description
        df.at[idx, 'type'] = info.type
        df.at[idx, 'traits'] = info.traits
        df.at[idx, 'cuisine'] = info.cuisine
        df.at[idx, 'price'] = info.price
        
    return df

# Process only the first 5 rows of the DataFrame
df = extract_structured_info(df)
df.to_csv('updated_copenhagen_curated_list.csv', index=False)

Response for row 0:
type='lunch' traits='hyggelig beliggenhed, dåsemad, overpris, dårlig service' cuisine='dåsemad' description='Café Slusen i Sydhavnen tilbyder en hyggelig beliggenhed, men maden, der primært består af dåsemad som tun, makrel og sardin, er overpriset. Selvom fisken var god, var den ikke særlig veltilberedt, og der var problemer med servicen. Det anbefales at nyde en øl eller kaffe i solen, men maden er ikke værd at betale for.' price='150 DKK'
Response for row 1:
type='dessert' traits='bæredygtigt, sundt, lækker isoplevelse, venligt personale' cuisine='is' description='Soltid er istid, og jeg skulle selvfølgelig prøve bananisen, som jeg har set alle mulige influencers guffe i sig, det er klart. @banana_cph laver is på frosne bananer, og det er både bæredygtigt og sundt. Jeg valgte et mix af ren bananis og den med lidt vegansk saltkaramel i, og fik dem til at fyre ekstra saltkaramel, kokosflager, ristede mandler og chokoladeknapper på toppen. Isoplevelsen var god, men 

Now we'll scrape information about all the restaurants to find what type of booking provider they use.

In [45]:
import time
import re
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs

def extract_booking_and_contact_info_selenium(df, max_entries=5):
    """
    For the first `max_entries` rows of the DataFrame, this function uses Selenium with explicit waits
    to load the restaurant's website (found via DuckDuckGo search) and scans the rendered page for:
    
      - Booking system keywords
      - The restaurant's website URL
      - A phone number
      - An email address

    If found, the function stores these values in the DataFrame columns:
      - booking_platform, website, phone, email
      
    If any information is not found, it defaults to "N/A".
    
    Parameters:
        df (pandas.DataFrame): Input DataFrame with a 'name' column.
        max_entries (int): Number of rows to process.
    
    Returns:
        pandas.DataFrame: The updated DataFrame.
    """
    # Keywords for booking systems and reservation links
    booking_systems = [
        "easytable", "opentable", "bookatable", "tock", "resy", "quandoo", 
        "tablein", "thefork", "/booking", "/reservation"
    ]
    reservation_keywords = ["reservation", "book", "table booking", "reserver", "reservering"]

    # Ensure the necessary columns exist
    for col in ["booking_platform", "website", "phone", "email"]:
        if col not in df.columns:
            df[col] = ""

    # Setup Selenium with headless Chrome and explicit wait
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--disable-gpu")
    driver = webdriver.Chrome(options=chrome_options)
    wait = WebDriverWait(driver, 15)  # wait up to 15 seconds for elements

    for idx in df.index[:max_entries]:
        restaurant_name = df.at[idx, "name"] if "name" in df.columns else ""
        booking_found = "N/A"
        website_found = "N/A"
        phone_found = "N/A"
        email_found = "N/A"

        if restaurant_name:
            query = f"{restaurant_name} restaurant Copenhagen"
            search_url = "https://duckduckgo.com/html/?q=" + query.replace(" ", "+")
            try:
                # Load the DuckDuckGo search results page
                driver.get(search_url)
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "a.result__a")))
                search_soup = BeautifulSoup(driver.page_source, "html.parser")
                result_link = search_soup.find("a", class_="result__a")
                if result_link and result_link.get("href"):
                    href = result_link.get("href")
                    # If the URL is a DuckDuckGo redirect, extract the actual URL from the "uddg" parameter.
                    if "uddg=" in href:
                        parsed = urlparse(href)
                        query_params = parse_qs(parsed.query)
                        if "uddg" in query_params:
                            website_found = query_params["uddg"][0]
                        else:
                            website_found = href
                    else:
                        # Ensure URL has a protocol
                        if href.startswith("//"):
                            website_found = "https:" + href
                        else:
                            website_found = href

                    print(f"Row {idx} - Found website URL: {website_found}")
                    
                    # Load the restaurant's website
                    driver.get(website_found)
                    wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
                    time.sleep(2)  # extra wait for dynamic content
                    website_source = driver.page_source.lower()

                    # --- Booking System Detection ---
                    # Strategy 1: Direct scan for booking system keywords in the page source.
                    for system in booking_systems:
                        if system in website_source:
                            booking_found = system
                            print(f"Row {idx} - Found booking system '{system}' via direct scan.")
                            break

                    # Strategy 2: Inspect reservation links if nothing found.
                    if booking_found == "N/A":
                        page_soup = BeautifulSoup(website_source, "html.parser")
                        for a in page_soup.find_all("a"):
                            link_text = a.get_text(strip=True).lower()
                            if any(keyword in link_text for keyword in reservation_keywords):
                                link_href = a.get("href", "").lower()
                                for system in booking_systems:
                                    if system in link_href:
                                        booking_found = system
                                        print(f"Row {idx} - Found booking system '{system}' via reservation link.")
                                        break
                            if booking_found != "N/A":
                                break

                    # --- Extract Phone and Email ---
                    # Use regex patterns to find a phone number and an email address.
                    phone_matches = re.findall(r'\+?\d[\d\s\-]{7,}\d', website_source)
                    if phone_matches:
                        phone_found = phone_matches[0].strip()
                    email_matches = re.findall(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', website_source)
                    if email_matches:
                        email_found = email_matches[0].strip()

                else:
                    print(f"Row {idx} - No result link found for query: {query}")
            except Exception as e:
                print(f"Error processing booking info for '{restaurant_name}': {e}")

        # Update the DataFrame with the scraped information
        df.at[idx, "booking_platform"] = booking_found
        df.at[idx, "website"] = website_found
        df.at[idx, "phone"] = phone_found
        df.at[idx, "email"] = email_found
        print(f"Row {idx} - Restaurant: {restaurant_name}, Booking System: {booking_found}, Website: {website_found}, Phone: {phone_found}, Email: {email_found}")

    driver.quit()
    return df

# Example usage:
# Process only the first 5 rows of the DataFrame and update the CSV.
df = extract_booking_and_contact_info_selenium(df, max_entries=5)
df.to_csv('updated_copenhagen_curated_list.csv', index=False)

Row 0 - Found website URL: https://restaurantguru.com/Cafe-Slusen-Copenhagen
Row 0 - Found booking system 'tock' via direct scan.
Row 0 - Restaurant: Cafe Slusen, Booking System: tock, Website: https://restaurantguru.com/Cafe-Slusen-Copenhagen, Phone: 1397034103, Email: wght@400..500
Row 1 - Found website URL: https://www.tripadvisor.com/Restaurant_Review-g189541-d4541300-Reviews-Banana_Joe-Copenhagen_Zealand.html
Row 1 - Restaurant: Banana, Booking System: N/A, Website: https://www.tripadvisor.com/Restaurant_Review-g189541-d4541300-Reviews-Banana_Joe-Copenhagen_Zealand.html, Phone: 923115816, Email: N/A
Row 2 - Found website URL: https://salon39.dk/
Row 2 - Found booking system '/booking' via direct scan.
Row 2 - Restaurant: Salon 39, Booking System: /booking, Website: https://salon39.dk/, Phone: 133-0-0-0, Email: book@salon39.dk
Row 3 - Found website URL: https://hosfischer.dk/
Row 3 - Found booking system 'tock' via direct scan.
Row 3 - Restaurant: Hos Fischer, Booking System: tock,